<a href="https://colab.research.google.com/github/skumarrms/lakebridge/blob/lakebrdige-code/Lakebridge_SNOWFLAKE_to_Databrikcs_Reconcile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS target_catalog.target_databricks_schema;
drop table target_catalog.target_databricks_schema.product;
CREATE TABLE IF NOT EXISTS target_catalog.target_databricks_schema.product (
    product_id INT,
    product_name STRING,
    price STRING,
    discount DECIMAL(5,3),
    offer DOUBLE,
    creation_date DATE,
    comment STRING
);

INSERT INTO target_catalog.target_databricks_schema.product  VALUES
(1, 'Laptop',     75000, 0.075, 5.5,  DATE '2024-07-01', 'Back-to-school offer'),
(2, 'Tablet',     30000, 0.050, 2.0,  DATE '2024-06-15', 'Limited stock'),
(3, 'Smartphone', 45000, 0.100, 3.0,  DATE '2024-07-10', 'New launch'),
(4, 'Monitor',    20000, 0.025, 1.0,  DATE '2024-05-30', 'Year-end clearance'),
(5, 'Keyboard',    2500, 0.010, 0.5,  DATE '2024-04-20', 'Regular item');

num_affected_rows,num_inserted_rows
5,5


In [ ]:
%sql
select * from target_catalog.target_databricks_schema.product;

product_id,product_name,price,discount,offer,creation_date,comment
1,Laptop,75000,0.075,5.5,2024-07-01,Back-to-school offer
2,Tablet,30000,0.050,2.0,2024-06-15,Limited stock
3,Smartphone,45000,0.100,3.0,2024-07-10,New launch
4,Monitor,20000,0.025,1.0,2024-05-30,Year-end clearance
5,Keyboard,2500,0.010,0.5,2024-04-20,Regular item


In [ ]:
%pip install git+https://github.com/databrickslabs/lakebridge
dbutils.library.restartPython()

  Cloning https://github.com/databrickslabs/lakebridge to /tmp/pip-req-build-6h9zh2o7
  Running command git clone --filter=blob:none --quiet https://github.com/databrickslabs/lakebridge /tmp/pip-req-build-6h9zh2o7
  Resolved https://github.com/databrickslabs/lakebridge to commit 91ee879a7cefad4c833f45cdee1e358568a786a0
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for databricks-labs-lakebridge: filename=databricks_labs_lakebridge-0.10.6-py3-none-any.whl size=200741 sha256=7ffe4f9a2f5d3f7e61681e4eb2120053d5a498c5076f4222bb1c525fafa6b59d
  Stored in directory: /tmp/pip-ephem-wheel-cache-i7c6ujae/wheels/6f/f1/22/a2829c4ad45bec3a8cc1a12346a6c4fc37799503f1768fa4c5
Successfully built databricks-l

In [ ]:
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig,
    TableRecon
)
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)
from databricks.labs.lakebridge.reconcile.execute import (
    recon,
    reconcile_aggregates
)
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

In [ ]:
from dataclasses import dataclass

@dataclass
class ReconcileConfig:
    data_source: str
    report_type: str
    secret_scope: str
    database_config: DatabaseConfig
    metadata_config: ReconcileMetadataConfig

In [ ]:
@dataclass
class DatabaseConfig:
    source_schema: str
    target_catalog: str
    target_schema: str
    source_catalog: str | None = None

In [ ]:
@dataclass
class ReconcileMetadataConfig:
    catalog: str = "lakebridge"
    schema: str = "reconcile"
    volume: str = "reconcile_volume"

In [ ]:
@dataclass
class Aggregate:
    agg_columns: list["price"]
    type: "min"
    group_by_columns: list[str] | None = None

In [ ]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)

reconcile_config = ReconcileConfig(
    data_source="snowflake",
    report_type="row",
    secret_scope="lakebridge_snowflake",  # Optional if authentication is already handled
    database_config=DatabaseConfig(
        source_catalog="SNOWFLAKE_LEARNING_DB",  # or your real catalog, if used
        source_schema="PUBLIC",
        target_catalog="target_catalog",  # or your real catalog, if used
        target_schema="target_databricks_schema"
    ),
    metadata_config=ReconcileMetadataConfig(
        catalog="lakebridge_metadata",
        schema="reconcile"
    )
)

In [ ]:
@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

In [ ]:
from databricks.labs.lakebridge.config import TableRecon
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    TableThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)

table_recon = TableRecon(
    source_catalog="SNOWFLAKE_LEARNING_DB",
    source_schema="PUBLIC",
    target_catalog="target_catalog",
    target_schema="target_schema",
    tables=[
        Table(
            source_name="product_prod",
            target_name="product",
            join_columns=["p_id"],
            drop_columns=["comment"],
            column_mapping=[
                ColumnMapping(source_name="p_id", target_name="product_id"),
                ColumnMapping(source_name="p_name", target_name="product_name")
            ],
            transformations=[
                Transformation(
                    column_name="creation_date",
                    source="creation_date",
                    target="to_date(creation_date,'yyyy-mm-dd')"
                )
            ],
            ##############
            aggregates=[
                Aggregate(
                    agg_columns=["price"],
                    type="min",
                    #group_by_columns=[]
                )
            ],
            #############
            column_thresholds=[
                ColumnThresholds(column_name="price", upper_bound="-50", lower_bound="50", type="float")
            ],
            table_thresholds=[
                TableThresholds(lower_bound="0%", upper_bound="5%", model="mismatch")
            ],
            jdbc_reader_options=JdbcReaderOptions(
                number_partitions=10,
                partition_column="p_id",
                lower_bound="0",
                upper_bound="10000000"
            ),
            filters=Filters(
                source="p_id > 0",
                target="product_id > 0"
            )
        )
    ]
)


In [ ]:
from databricks.labs.lakebridge import __version__
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.reconcile.execute import recon
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

ws = WorkspaceClient(product="lakebridge", product_version=__version__)

try:
  result = recon(
            ws = ws,
            spark = spark, # notebook spark session
            table_recon = table_recon, # previously created
            reconcile_config = reconcile_config # previously created
          )
  print(result.recon_id)
  print(result)
  print("***************************")
except ReconciliationException as e:
    recon_id = e.reconcile_output.recon_id
    print(f" Failed : {recon_id}")

    print(e)
    print("***************************")
except Exception as e:
    print(e.with_traceback)
    raise e
    print(f"Exception : {str(e)}")
    print("***************************")


04:37:14  WARNING [d.l.l.r.connectors.snowflake] pem_private_key not found. Checking for sfPassword
04:37:25  WARNING [d.l.l.r.connectors.data_source] Runtime exception occurred while fetching data using SELECT LOWER(SHA2(CONCAT(creation_date, COALESCE(TRIM(discount), '_null_recon_'), COALESCE(TRIM(offer), '_null_recon_'), COALESCE(TRIM(p_id), '_null_recon_'), COALESCE(TRIM(p_name), '_null_recon_')), 256)) AS hash_value_recon, creation_date AS creation_date, discount AS discount, offer AS offer, p_id AS p_id, p_name AS p_name FROM SNOWFLAKE_LEARNING_DB.PUBLIC.product_prod WHERE p_id > 0 : The input query contains unsupported data source(s). Only csv, json, avro, delta, kafka, parquet, orc, text, unity_catalog, binaryFile, xml, excel, simplescan, iceberg, mysql, postgresql, sqlserver, redshift, snowflake, sqldw, databricks, bigquery, oracle, salesforce, salesforce_data_cloud, teradata, workday_raas, mongodb, h2 data sources are supported on serverless compute, and only csv, json, avro, 

 Failed : 99475bfc-77aa-4b16-ba1b-8672d0b92262
(' Reconciliation failed for one or more tables. Please check the recon metrics for more details. **reconcile** failed.', ReconcileOutput(recon_id='99475bfc-77aa-4b16-ba1b-8672d0b92262', results=[ReconcileTableOutput(target_table_name='target_catalog.target_databricks_schema.product', source_table_name='SNOWFLAKE_LEARNING_DB.PUBLIC.product_prod', status=StatusOutput(row=None, column=None, schema=None, aggregate=None), exception_message='Runtime exception occurred while fetching data using SELECT LOWER(SHA2(CONCAT(creation_date, COALESCE(TRIM(discount), _null_recon_), COALESCE(TRIM(offer), _null_recon_), COALESCE(TRIM(p_id), _null_recon_), COALESCE(TRIM(p_name), _null_recon_)), 256)) AS hash_value_recon, creation_date AS creation_date, discount AS discount, offer AS offer, p_id AS p_id, p_name AS p_name FROM SNOWFLAKE_LEARNING_DB.PUBLIC.product_prod WHERE p_id > 0 : The input query contains unsupported data source(s). Only csv, json, avro, 

In [ ]:
%sql
select * from lakebridge_metadata.reconcile.main where recon_id = 'e450f215-e0f0-4511-bcbf-5b28c0c4d116';

recon_table_id,recon_id,source_type,source_table,target_table,report_type,operation_name,start_ts,end_ts
6106084016888052679,e450f215-e0f0-4511-bcbf-5b28c0c4d116,Snowflake,"List(SNOWFLAKE_LEARNING_DB, PUBLIC, product_prod)","List(target_catalog, target_databricks_schema, product)",schema,reconcile,2025-08-04T04:18:28.014Z,2025-08-04T04:18:36.091Z


In [ ]:
%sql
select * from lakebridge_metadata.reconcile.details where recon_table_id = 6106084016888052679;

recon_table_id,recon_type,status,data,inserted_ts
6106084016888052679,schema,false,"List(Map(source_column -> p_id, source_datatype -> number(38,0), databricks_column -> product_id, databricks_datatype -> int, is_valid -> false), Map(source_column -> p_name, source_datatype -> varchar(16777216), databricks_column -> product_name, databricks_datatype -> string, is_valid -> true), Map(source_column -> price, source_datatype -> float, databricks_column -> price, databricks_datatype -> string, is_valid -> false), Map(source_column -> discount, source_datatype -> number(5,3), databricks_column -> discount, databricks_datatype -> decimal(5,3), is_valid -> true), Map(source_column -> offer, source_datatype -> float, databricks_column -> offer, databricks_datatype -> double, is_valid -> true), Map(source_column -> creation_date, source_datatype -> date, databricks_column -> creation_date, databricks_datatype -> date, is_valid -> true))",2025-08-04T04:18:40.395Z


In [ ]:
%sql
select * from lakebridge_metadata.reconcile.metrics where recon_table_id = 6106084016888052679;

recon_table_id,recon_metrics,run_metrics,inserted_ts
6106084016888052679,"List(null, null, false)","List(false, kaledhiraj7777@gmail.com, )",2025-08-04T04:18:38.176Z


In [ ]:
df = spark.read \
    .format("snowflake") \
    .option("sfURL", "UVGMTRL-CQ23371.snowflakecomputing.com") \
    .option("sfUser", "DHIRAJKALE") \
    .option("sfPassword", "Dhirajka131162@") \
    .option("sfDatabase", "SNOWFLAKE_LEARNING_DB") \
    .option("sfSchema", "PUBLIC") \
    .option("sfWarehouse", "SNOWFLAKE_LEARNING_WH") \
    .option("query", "SELECT * FROM product_prod LIMIT 10") \
    .load()
df.show()

+----+----------+-------+--------+-----+-------------+--------------------+
|P_ID|    P_NAME|  PRICE|DISCOUNT|OFFER|CREATION_DATE|             COMMENT|
+----+----------+-------+--------+-----+-------------+--------------------+
|   1|    Laptop|75000.0|   0.075|  5.1|   2024-07-01|Back-to-school offer|
|   2|    Tablet|30000.0|   0.050|  2.2|   2024-06-15|       Limited stock|
|   3|Smartphone|45000.0|   0.100|  3.1|   2024-07-10|          New launch|
|   4|   Monitor|20000.0|   0.025|  1.0|   2024-05-30|  Year-end clearance|
|   5|  Keyboard| 2500.0|   0.010|  0.5|   2024-04-20|        Regular item|
+----+----------+-------+--------+-----+-------------+--------------------+

